In [ ]:
import os


In [ ]:
meme_file_path = "../motif_databases/CISBP-RNA/Homo_sapiens.meme"

ev_rbps = [
    '(RBM38)_(Mus_musculus)_(RBD_0.99)', 'A2BP1', 'CSDA', 'ELAVL2', 'HNRNPC', 'SNRPA', 'TARDBP', 'hnRNPK',
    '(Elavl2)_(Homo_sapiens)_(RBD_1.00)', '(Hnrnpc)_(Homo_sapiens)_(RBD_1.00)', '(Rbfox1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Snrpa)_(Homo_sapiens)_(RBD_0.98)', '(Tardbp)_(Homo_sapiens)_(RBD_0.92)', '(Ybx1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Zc3h10)_(Homo_sapiens)_(RBD_1.00)', 'Rbm38', 'YBX1'
]

cyto_rbps = [
    '(RBM45)_(Mus_musculus)_(RBD_0.94)', 'CPEB2', 'FMR1', 'HNRNPL', 'LIN28A', 'NOVA2', 'PABPN1', 'PPRC1', 
    'RBM3', 'RBM8A', 'SAMD4A', 'SRSF7', 'STAR-PAP', 'hnRNPLL',
    '(Cpeb2)_(Homo_sapiens)_(RBD_1.00)', '(Fmr1)_(Homo_sapiens)_(RBD_0.97)', '(Hnrpll)_(Homo_sapiens)_(RBD_0.99)', 
    '(Lin28a)_(Homo_sapiens)_(RBD_0.99)', '(Lin28b)_(Homo_sapiens)_(RBD_0.83)', '(Nova1)_(Homo_sapiens)_(RBD_0.91)', 
    '(Pabpn1)_(Homo_sapiens)_(RBD_1.00)', '(Pprc1)_(Homo_sapiens)_(RBD_1.00)', '(Rbm3)_(Homo_sapiens)_(RBD_0.99)', 
    '(Rbm8a)_(Homo_sapiens)_(RBD_1.00)', '(Samd4)_(Homo_sapiens)_(RBD_0.95)', '(Tut1)_(Homo_sapiens)_(RBD_0.85)', 
    'Rbm45', 'LIN28B', 'NOVA1'
]

shared_rbps = [
    '(HNRNPR)_(Gallus_gallus)_(RBD_0.97)', '(PCBP3)_(Mus_musculus)_(RBD_1.00)', 'CPEB4', 'HNRNPCL1', 'MBNL1', 
    'PCBP1', 'PCBP2', 'PTBP1', 'RALY', 'ROD1', 'TIA1', 'U2AF2', 'YB-1', 'ZC3H10',
    '(Cpeb4)_(Homo_sapiens)_(RBD_1.00)', '(Csda)_(Homo_sapiens)_(RBD_1.00)', '(Hnrnpk)_(Homo_sapiens)_(RBD_1.00)', 
    '(Mbnl1)_(Homo_sapiens)_(RBD_1.00)', '(Pcbp2)_(Homo_sapiens)_(RBD_1.00)', '(Ptbp1)_(Homo_sapiens)_(RBD_0.96)', 
    '(Raly)_(Homo_sapiens)_(RBD_0.97)', '(Rod1)_(Homo_sapiens)_(RBD_0.80)', '(Tia1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Tial1)_(Homo_sapiens)_(RBD_0.89)', '(U2af2)_(Homo_sapiens)_(RBD_1.00)', 'Pcbp1', 'Pcbp3', 'TIAL1'
]

def get_consensus(matrix_lines, alphabet="ACGU"):
    consensus = ""
    for line in matrix_lines:
        try:
            probs = list(map(float, line.strip().split()))
            if not probs: continue
            max_idx = probs.index(max(probs))
            consensus += alphabet[max_idx]
        except ValueError:
            continue
    return consensus

def extract_meme_data(input_file, target_list, output_file, group_name):
    targets_upper = [t.upper() for t in target_list]
    
    with open(input_file, 'r') as f:
        lines = f.readlines()

    header_lines = []
    motif_blocks = []
    found_count = 0
    
    in_header = True
    current_motif_lines = []
    keep_current_motif = False
    current_motif_name = ""
    matrix_lines = []
    in_matrix = False

    for line in lines:
        if line.startswith("MOTIF"):
            in_header = False
            if keep_current_motif:
                motif_blocks.append("".join(current_motif_lines))
                consensus = get_consensus(matrix_lines)
                print(f"[{group_name}] found: {current_motif_name.ljust(45)} | Consensus: {consensus}")
                found_count += 1
            
            current_motif_lines = [line]
            matrix_lines = []
            in_matrix = False
            
            line_upper = line.upper()
            keep_current_motif = any(t in line_upper for t in targets_upper)
            
            if keep_current_motif:
                parts = line.strip().split()
                current_motif_name = parts[-1] if len(parts) > 1 else parts[0]
                
        elif in_header:
            header_lines.append(line)
        else:
            current_motif_lines.append(line)
            if line.startswith("letter-probability matrix:"):
                in_matrix = True
            elif line.startswith("URL") or line.strip() == "":
                in_matrix = False
            elif in_matrix:
                matrix_lines.append(line)

    if keep_current_motif:
        motif_blocks.append("".join(current_motif_lines))
        consensus = get_consensus(matrix_lines)
        print(f"[{group_name}] found: {current_motif_name.ljust(45)} | Consensus: {consensus}")
        found_count += 1

    with open(output_file, 'w') as out_f:
        out_f.writelines(header_lines)
        out_f.writelines(motif_blocks)
    
    print(f"Summary: {group_name} matched {found_count} motif blocks.")
    print(f"--- Results saved to {output_file} ---\n")

print("Extracting merged motif information from the database, including human and mouse supplemental motifs...\n")
extract_meme_data(meme_file_path, ev_rbps, "./meme_files/EV_specific_motifs.meme", "EV-specific")
extract_meme_data(meme_file_path, cyto_rbps, "./meme_files/Cyto_specific_motifs.meme", "Cyto-specific")
extract_meme_data(meme_file_path, shared_rbps, "./meme_files/Shared_motifs.meme", "Shared RBPs")